# 02 - Data Cleaning and Merging
In this notebook, we will clean the raw datasets, handle missing values, remove duplicates, convert date columns into proper formats, standardize formats, and merge them into a single consolidated dataset. Finally, we will save the cleaned output to the `data/processed` folder.

In [ ]:
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

RAW_DIR = Path('../data/raw')
PROCESSED_DIR = Path('../data/processed')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

## 1. Process NAV History Data
We will combine all individual NAV files into a single dataframe, standardize the date, and drop missing values and duplicates.

In [ ]:
nav_files = list(RAW_DIR.glob('*_nav.csv'))
nav_dfs = []

for f in nav_files:
    # Extract fund_id from filename
    fund_id = f.stem.split('_')[0]
    df = pd.read_csv(f)
    # Standardize column names
    df.columns = df.columns.str.lower().str.strip()
    df['fund_id'] = str(fund_id)
    
    # Convert dates to datetime
    if 'date' in df.columns:
        df['date'] = pd.to_datetime(df['date'], dayfirst=True, errors='coerce')
    
    nav_dfs.append(df)

combined_nav = pd.concat(nav_dfs, ignore_index=True)
# Clean Data: drop NAs and duplicates
combined_nav.dropna(subset=['date', 'nav'], inplace=True)
combined_nav.drop_duplicates(subset=['fund_id', 'date'], inplace=True)
combined_nav.sort_values(by=['fund_id', 'date'], inplace=True)

combined_nav.to_csv(PROCESSED_DIR / 'cleaned_nav.csv', index=False)
print(f'Cleaned NAV Data Shape: {combined_nav.shape}')
combined_nav.head()

## 2. Process AUM and Investor Transactions
Next, we load the available AUM and transaction data, format their dates, and aggregate daily inflows.

In [ ]:
aum_path = RAW_DIR / 'aum.csv'
if aum_path.exists():
    aum_df = pd.read_csv(aum_path)
    aum_df.columns = aum_df.columns.str.lower().str.strip()
    if 'aum_id' in aum_df.columns:
        aum_df.rename(columns={'aum_id': 'id'}, inplace=True)
    aum_df['date'] = pd.to_datetime(aum_df['date'], dayfirst=True, errors='coerce')
    aum_df['fund_id'] = aum_df['fund_id'].astype(str)
    aum_df.dropna(subset=['date', 'fund_id'], inplace=True)
    aum_df.drop_duplicates(subset=['fund_id', 'date'], inplace=True)
else:
    aum_df = pd.DataFrame()

trans_path = RAW_DIR / 'investor_transactions.csv'
if trans_path.exists():
    trans_df = pd.read_csv(trans_path)
    trans_df.columns = trans_df.columns.str.lower().str.strip()
    trans_df['date'] = pd.to_datetime(trans_df['transaction_date'], dayfirst=True, errors='coerce')
    trans_df['fund_id'] = trans_df['fund_id'].astype(str)
    trans_df.dropna(subset=['date', 'fund_id'], inplace=True)
    # Aggregate transactions by fund and date
    agg_trans = trans_df.groupby(['fund_id', 'date'], as_index=False)['amount'].sum()
    agg_trans.rename(columns={'amount': 'daily_inflow'}, inplace=True)
else:
    agg_trans = pd.DataFrame()

print(f'AUM Data Shape: {aum_df.shape}')
print(f'Aggregated Transactions Shape: {agg_trans.shape}')

## 3. Merge Datasets
We merge the `combined_nav` baseline with our `aum_df` and `agg_trans` using `fund_id` and `date` as keys. Since NAV is daily and AUM might be less frequent, we can use forward filling for AUM.

In [ ]:
merged_df = combined_nav.copy()

if not aum_df.empty:
    merged_df = pd.merge(merged_df, aum_df[['fund_id', 'date', 'aum']], on=['fund_id', 'date'], how='left')
    # Forward fill AUM for continuous daily records
    merged_df['aum'] = merged_df.groupby('fund_id')['aum'].ffill()

if not agg_trans.empty:
    merged_df = pd.merge(merged_df, agg_trans[['fund_id', 'date', 'daily_inflow']], on=['fund_id', 'date'], how='left')
    # Fill missing inflow days with 0
    merged_df['daily_inflow'] = merged_df['daily_inflow'].fillna(0)

# Save final dataset
merged_path = PROCESSED_DIR / 'merged_dataset.csv'
merged_df.to_csv(merged_path, index=False)
print(f'Final Merged Dataset Shape: {merged_df.shape}')
merged_df.head()